In [ ]:
from sagemaker.pytorch import PyTorch
from sagemaker import Session
from sagemaker import get_execution_role
from sagemaker.utils import unique_name_from_base
from sagemaker.inputs import TrainingInput
import os


## SET THIS
project_name = ""
run_name = unique_name_from_base(project_name)
print( run_name )

if not project_name:
    raise ValueError("MUST SET THE PROJECT NAME!!")

# If running notebook inside a SageMaker Notebook or Studio instance,
# can retrieve the execution role associated with the instance
# role = get_execution_role()
role = ""

# Copy ARN of the MLFlow Server
mlflow_arn = ""


# Set the Paths to the Images and Masks and path to the split file
split_file = "s3://wwps-llnl-model-dev/Dataset-2K/split_file_dataset2k.json"
train_image = "s3://wwps-llnl-model-dev/Dataset-2K/images"
train_mask = "s3://wwps-llnl-model-dev/Dataset-2K/masks"

image_input = TrainingInput(train_image, input_mode="FastFile", s3_data_type="S3Prefix", distribution="FullyReplicated")
mask_input = TrainingInput(train_mask, input_mode="FastFile", s3_data_type="S3Prefix", distribution="FullyReplicated")

# Set up checkpointing s3 path. Checkpointing will periodically upload the given folder in the training
# container to S3. This will allow model training resume and enable the use of spot instances.
# Update bucket name to use a specific bucket for checkpoint paths
checkpoint_bucket = Session().default_bucket()
checkpoint_s3_uri = f"s3://{checkpoint_bucket}/{run_name}/checkpoints"
local_checkpoint_path = "/opt/ml/checkpoints"

# If launching multiple jobs within a short period of time utilizing the same instance types
# Update this value to a time in seconds to create a Warm Pool. This will enable faster 
# launching of subsequent training jobs after the first. NOTE: Costs are associated with keeping a warm pool alive
# More details here: https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html
warm_pool_duration = 0


hyperparameters = {
    "num_workers": 8,#8,
    "model_name": "UNet",
    # Point to a .ckpt file in S3 to load those weights prior to training starting.
    # Otherwise, will train model from scratch. Note, checkpoints in checkpoint directory
    # will overwrite these on model training resume.
    # "pretrained_weights": "",
    # Note, with Pytorch Lightning, this is a per GPU batch count. PL will create a batch per device
    # behind the scenes. No need to adjust if changing instances with different gpu counts. 
    # Recommend adjust if changes instances that have different GPU memory (e.g., p3.* -> p3dn.* -> p4d.* -> p4de.*)
    "batch_size": 4,  # Reduced from 4 to avoid OOM during DDP initialization with 8 GPUs
    "epochs": 4,
    "learning_rate": 1e-3,
    "lr_schedular": "cosine_warmup",
    "image_mode": "grayscale",
    "split_file": split_file,
    # Which precision to use for training
    "precision": "16-mixed",
    # Data Augmentation Parameters
    "image_size": 1200,
    "use_random_resize": "False",
    "use_random_rotation": "False",
    "color_jitter": 0.1,
    "center_crop": "True",
    "center_crop_size": 1200,
    "test_image_size": 1200,
    "test_batch_size": 1,
    "run_name": run_name,
    "project_name": project_name,
    "checkpoint_dir": local_checkpoint_path,
    # Metadata Test Analysis
    # Provide a s3 uri to a csv containing metadata to perform metadata test results analysis.
    # A parquet file with per image test results will always be output in the output.tar.gz file for the training run.
    # If metadata provided, metadata values are included in the dataframe per image.
    # "metadata_file": '""',
}

estimator = PyTorch(
    entry_point="train.py",
    # Point to the repo root one level up,
    # this notebook is at {repo}/notebooks/LaunchTrainingJob.ipynb
    source_dir="src/",  
    role=role,
    framework_version="2.7",
    py_version="py312",
    instance_count=1,
    instance_type= 'ml.g6.12xlarge', #'ml.p4d.24xlarge', # 'ml.g5.4xlarge', #  #'ml.g4dn.4xlarge', #'ml.p3.8xlarge', #'ml.p3.16xlarge', #'ml.p4d.24xlarge', # 80GB A100 GPU   "ml.g4dn.4xlarge", 
    train_volume_size = 125,
    hyperparameters=hyperparameters, 
    # Default input_mode is "File" mode which will download a copy of all contents in S3
    # locally prior to training. "FastFile" will instead stream files from S3 as they are
    # requested. This essentially makes S3 a mounted drive to stream the data from.
    # More information can be found here: 
    # https://docs.aws.amazon.com/sagemaker/latest/dg/model-access-training-data.html
    input_mode="FastFile",
    # metric_definitions=metric_def,
    # SM will continously backup checkpoints from the local path to the s3 uri
    # In case of model iterruption (e.g., when using spot), the checkpoints on s3 are
    # are placed into the local path on resume
    checkpoint_s3_uri=checkpoint_s3_uri,
    checkpoint_local_path=local_checkpoint_path,
    # NOTE: If launching job from outside SageMaker Domain
    # tensorboard_ouptut_config may cause errors when running the training job. Investigating cause and remediation.
    # If job fails with "internal error", comment out below line and relaunch job.    
    environment={
        "MLFLOW_TRACKING_URI": mlflow_arn,
        "MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING": "true",
        "FI_EFA_FORK_SAFE": "1"
    },
    #tensorboard_output_config=tensorboard_output_config,
    # keep_alive_period_in_seconds=warm_pool_duration,
)

# Set wait=True to show training logs in notebook.
estimator.fit(
    job_name=run_name,
    inputs={"train_image": image_input, "train_mask": mask_input}, wait=False,
)